In [1]:
import json
import random

random.seed(42) # 设置随机种子以保证结果可重复

# 1. 结构化读取所有数据
data_by_view_diff = {
    'edge': {'simple': [], 'medium': [], 'hard': []},
    'path': {'simple': [], 'medium': [], 'hard': []}
}

with open('training_data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        view = item.get('view')
        difficulty = item.get('difficulty')
        if view in data_by_view_diff and difficulty in data_by_view_diff[view]:
            data_by_view_diff[view][difficulty].append(item)

# 2. 采样函数（如果数据量不足时，会在同类中允许重复采样，但目前单个文件的需求量均小于库存总量）
def get_samples(view, difficulty, k):
    pool = data_by_view_diff[view][difficulty]
    if k <= len(pool):
        return random.sample(pool, k)
    else:
        print(f"Warning: Not enough {view}-{difficulty} data. Needed {k}, but only have {len(pool)}. Sampling with replacement.")
        return random.choices(pool, k=k)

# 3. 构造 edge-only (1000条, simple:medium:hard = 44:3:53)
edge_only = []
edge_only.extend(get_samples('edge', 'simple', 440))
edge_only.extend(get_samples('edge', 'medium', 30))
edge_only.extend(get_samples('edge', 'hard', 530))
random.shuffle(edge_only)

with open('edge-only.jsonl', 'w', encoding='utf-8') as f:
    for item in edge_only:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# 4. 构造 path-only (1000条, simple:medium:hard = 44:3:53)
path_only = []
path_only.extend(get_samples('path', 'simple', 440))
path_only.extend(get_samples('path', 'medium', 30))
path_only.extend(get_samples('path', 'hard', 530))
random.shuffle(path_only)

with open('path-only.jsonl', 'w', encoding='utf-8') as f:
    for item in path_only:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# 5. 构造 mixed (1000条, path:edge = 4:6)
# 其中 path=400条, edge=600条，均符合 44:3:53 的难度比例
mixed = []
# path 比例计算 (400条)
mixed.extend(get_samples('path', 'simple', int(400 * 0.44)))  # 176
mixed.extend(get_samples('path', 'medium', int(400 * 0.03)))  # 12
mixed.extend(get_samples('path', 'hard', int(400 * 0.53)))    # 212
# edge 比例计算 (600条)
mixed.extend(get_samples('edge', 'simple', int(600 * 0.44)))  # 264
mixed.extend(get_samples('edge', 'medium', int(600 * 0.03)))  # 18
mixed.extend(get_samples('edge', 'hard', int(600 * 0.53)))    # 318
random.shuffle(mixed)

with open('mixed.jsonl', 'w', encoding='utf-8') as f:
    for item in mixed:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("三个数据集文件已成功生成：edge-only.jsonl, path-only.jsonl, mixed.jsonl")


三个数据集文件已成功生成：edge-only.jsonl, path-only.jsonl, mixed.jsonl
